In [24]:
import pandas as pd
funda = pd.read_csv(r"C:\Users\quent\Documents\SKEMA\COURSES\Commodities\projects\3\1. Raw Data\funda.csv")

In [25]:
funda

,gasDayStart,gasInStorage,workingGasVolume,full,trend,injection,withdrawal,netWithdrawal,injectionCapacity,withdrawalCapacity,consumption,consumptionFull,status
0,2011-01-01,440.3870,617.5104,71.32,0.00,147.02,1536.0,1389.0,5392.92,10316.05,3530.0,12.48,C
1,2011-01-02,438.4398,617.5102,71.00,-0.32,84.05,2091.4,2007.3,5392.84,10316.05,3530.0,12.42,C
2,2011-01-03,435.3212,617.5099,70.50,-0.51,110.60,3323.2,3212.6,5392.74,10316.04,3530.0,12.33,C
3,2011-01-04,431.5952,617.5105,69.89,-0.60,44.25,3829.5,3785.2,5392.62,10316.06,3530.0,12.23,C
4,2011-01-05,428.0276,617.5105,69.32,-0.58,62.45,3640.9,3578.5,5392.47,10316.05,3530.0,12.13,C
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5755,2026-10-04,822.9733,1132.3922,72.68,0.27,3299.52,249.3,-3050.2,11767.02,19600.15,3519.0,23.39,C
5756,2026-10-05,824.7975,1132.3922,72.84,0.16,2275.87,449.5,-1826.3,11767.86,19601.25,3519.0,23.44,C
5757,2026-10-06,826.2603,1132.3922,72.97,0.13,1795.84,509.5,-1286.4,11767.86,19601.25,3519.0,23.48,C
5758,2026-10-07,828.0342,1132.3922,73.12,0.16,2169.86,392.7,-1777.1,11767.86,19601.25,3519.0,23.53,E


In [26]:
funda["agsi_estimated"] = 0
funda.loc[funda["status"] == "E", "agsi_estimated"] = 1 # 1 if the value is estimated, 0 if it is confirmed


In [27]:
funda["gasDayStart"] = pd.to_datetime(funda["gasDayStart"])

feb29 = funda["gasDayStart"].dt.month.eq(2) & funda["gasDayStart"].dt.day.eq(29)
key = funda["gasDayStart"].dt.strftime("%m-%d").mask(feb29, "02-28")  # 29 Feb uses the 28 Feb normal
year = funda["gasDayStart"].dt.year

# 29 February stays out of the 28 February average
hist = funda.loc[~feb29, ["gasDayStart", "full"]].copy()
hist["key"] = hist["gasDayStart"].dt.strftime("%m-%d")
hist["year"] = hist["gasDayStart"].dt.year
lookup = hist.set_index(["key", "year"])["full"]

prior = pd.DataFrame({
    lag: pd.MultiIndex.from_arrays([key, year - lag]).map(lookup)
    for lag in range(1, 6)
})
funda["full_norm5y"] = prior.mean(axis=1, skipna=False)  # five preceding years, or missing
funda["storage_dev"] = funda["full"] - funda["full_norm5y"]
funda["storage_dev_sq"] = funda["storage_dev"] ** 2


In [28]:
funda["net_inj"] = -funda["netWithdrawal"] / 1000  # TWh/d, positive when injecting
funda["inj_util"] = funda["injection"] / funda["injectionCapacity"]
funda["wd_util"] = funda["withdrawal"] / funda["withdrawalCapacity"]


In [29]:
funda = funda.drop(columns=[
    "trend", "consumption", "consumptionFull", "status",
    "gasInStorage", "workingGasVolume",
    "injection", "withdrawal", "netWithdrawal",
    "injectionCapacity", "withdrawalCapacity",
])


In [30]:
funda = funda.sort_values("gasDayStart")
cols = funda.columns.drop("gasDayStart")
funda[cols] = funda[cols].shift(2) # gas day D is usable at D+2

In [31]:
funda.to_csv("funda_processed.csv", index=False)